In [20]:
import pandas as pd
import numpy as np
from datetime import time
import statsmodels.api as sm
import datetime as dt
start_date = "2005-03-01"
end_date = "2006-03-01"
df = pd.read_csv('data/SPY_15min_2002-01_to_2006-08.csv')
df2 = pd.read_csv('data/SPY_15min_2020-01_to_2022-01.csv')


#df['datetime'] = df['datetime'].dt.tz_localize('US/Eastern')
df['datetime'] = pd.to_datetime(df['Unnamed: 0'], utc=True)
print(df['Unnamed: 0'].head())
print(df['Unnamed: 0'].dtype)


df.set_index('datetime', inplace=True)
df.index = pd.DatetimeIndex(df.index)  # This ensures time-aware index
df = df.asfreq('15min')  # Ensure the index is at 15-minute frequency
df.index = df.index.tz_convert("Europe/Berlin")  # Convert to CET timezone
df = df.between_time("15:30", "22:00")
df = df[df.index.weekday < 5] # Keep only Monday (0) through Friday (4)

df["y^2"] = (df["close"].apply(lambda x: np.log(x)).diff()) ** 2
df = df.loc[pd.Timestamp(start_date).tz_localize("Europe/Berlin"):pd.Timestamp(end_date).tz_localize("Europe/Berlin")]

df.drop(columns=['Unnamed: 0'], inplace=True)
df.interpolate(method='time', inplace=True)


df["day_of_week"] = df.index.dayofweek
df["hour_of_day"] = df.index.hour

day_dummies = pd.get_dummies(df["day_of_week"], prefix="day", drop_first=True)
hour_dummies = pd.get_dummies(df["hour_of_day"], prefix="hour", drop_first=True)




df = pd.concat([df, day_dummies, hour_dummies], axis=1)
df


0    2002-01-02 09:30:00-05:00
1    2002-01-02 09:45:00-05:00
2    2002-01-02 10:00:00-05:00
3    2002-01-02 10:15:00-05:00
4    2002-01-02 10:30:00-05:00
Name: Unnamed: 0, dtype: object
object


,open,high,low,close,volume,y^2,day_of_week,hour_of_day,day_1,day_2,day_3,day_4,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22
datetime,,,,,,,,,,,,,,,,,,,
2005-03-01 15:30:00+01:00,83.1154,83.4113,83.1154,83.3769,3869200.0,1.779731e-05,1,15,True,False,False,False,False,False,False,False,False,False,False
2005-03-01 15:45:00+01:00,83.3769,83.4044,83.2805,83.3494,2046100.0,1.088221e-07,1,15,True,False,False,False,False,False,False,False,False,False,False
2005-03-01 16:00:00+01:00,83.3494,83.5971,83.2874,83.4870,5279800.0,2.720918e-06,1,16,True,False,False,False,True,False,False,False,False,False,False
2005-03-01 16:15:00+01:00,83.4870,83.5145,83.4044,83.4113,2816800.0,8.229017e-07,1,16,True,False,False,False,True,False,False,False,False,False,False
2005-03-01 16:30:00+01:00,83.4113,83.5489,83.3906,83.4870,1562300.0,8.229017e-07,1,16,True,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006-02-28 21:00:00+01:00,89.8593,89.9154,89.7263,89.7543,3240400.0,1.190879e-06,1,21,True,False,False,False,False,False,False,False,False,True,False
2006-02-28 21:15:00+01:00,89.7543,89.9154,89.7473,89.9014,1836600.0,2.681659e-06,1,21,True,False,False,False,False,False,False,False,False,True,False
2006-02-28 21:30:00+01:00,89.9014,89.9854,89.8453,89.9644,1822900.0,4.907315e-07,1,21,True,False,False,False,False,False,False,False,False,True,False


In [56]:
#todo add overnight & over weekend dummmies
def safe_add(column: pd.Series, value: int): # Expects a datetimeIndex
    if not pd.api.types.is_datetime64_any_dtype(column):
        raise TypeError("Excpected a Series with DatetimeIndex.")
    # Check if the index is timezone-aware
    if column.dt.tz is None:
        raise ValueError("DatetimeIndex must be timezone-aware.")
    # Cast to ET timezone for timezone aware consistency
    column = column.dt.tz_convert("America/New_York")
    # Check whether value is outside of market open hours
    hypothetical_values = column + pd.Timedelta(minutes=value)

    mask_before_open = (hypothetical_values.dt.time < dt.time(9,30))
    mask_during_open = (dt.time(9,30) <= hypothetical_values.dt.time) & (hypothetical_values.dt.time <= dt.time(16,0))
    mask_after_open = (hypothetical_values.dt.time > dt.time(16,0))

    date_of_action = column.dt.normalize() # Extract date and set hours to 0:00
    output = column.copy()
    output[mask_during_open] = output[mask_during_open] + pd.Timedelta(minutes=value)
    if value >= 0: # Safe add
        output[mask_before_open] = date_of_action[mask_before_open] + pd.Timedelta(hours=9, minutes=30) + pd.Timedelta(minutes=value)
        output[mask_after_open] = date_of_action[mask_after_open] + pd.Timedelta(days=1) + pd.Timedelta(hours=9, minutes=30) + pd.Timedelta(minutes=value)
    else: # Safe subtract
        #todo check what happens when input to timedelta is negative
        output[mask_before_open] = date_of_action[mask_before_open] - pd.Timedelta(days=1) + pd.Timedelta(hours=16) + pd.Timedelta(minutes=value)
        output[mask_after_open] = date_of_action[mask_after_open] + pd.Timedelta(hours=16, minutes=value)

    return output # Caution: returns series with ET timezone


df_news = pd.read_csv('data/WhatMovesMarkets_eventdatabase.csv')
df_news["eventstart_CET"] = pd.to_datetime(df_news["eventstart_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
df_news["eventend_CET"] = pd.to_datetime(df_news["eventend_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
#df_news["eventstart_CET"] = df_news["eventstart_CET"].dt.floor("15min")  # Optional: Round down to the nearest 15 minutes (not mentioned in paper)

# Correct time window creation according to paper

event_end_mask = df_news["eventend_CET"].notna()

# Only assign to rows where event_end_mask is True
df_news.loc[event_end_mask, "window_start"] = (
    safe_add(df_news.loc[event_end_mask, "eventstart_CET"], -20)
    .dt.tz_convert("Etc/GMT-1")
)

df_news.loc[event_end_mask, "window_end"] = (
    safe_add(df_news.loc[event_end_mask, "eventend_CET"], 20)
    .dt.tz_convert("Etc/GMT-1")
)

# Only assign to rows where event_end_mask is False
df_news.loc[~event_end_mask, "window_start"] = (
    safe_add(df_news.loc[~event_end_mask, "eventstart_CET"], -15)
    .dt.tz_convert("Etc/GMT-1")
)

df_news.loc[~event_end_mask, "window_end"] = (
    safe_add(df_news.loc[~event_end_mask, "eventstart_CET"], 30)
    .dt.tz_convert("Etc/GMT-1")
)


df_news


,eventstart,eventend,name,type,subtype,subsubtype,description,source,scheduled,eventstart_CET,eventend_CET,window_start,window_end
0,01.03.2002 01:00:00,NaT,FI Consumer Confidence,Macro Release,FI,Consumer Confidence,NaN,Bloomberg,1,2002-03-01 07:00:00+01:00,NaT,2002-02-28 21:45:00+01:00,2002-03-01 16:00:00+01:00
1,01.03.2002 01:30:00,NaT,CH CPI,Macro Release,CH,CPI,NaN,Bloomberg,1,2002-03-01 07:30:00+01:00,NaT,2002-02-28 21:45:00+01:00,2002-03-01 16:00:00+01:00
2,01.03.2002 03:00:00,NaT,IT CPI,Macro Release,IT,CPI,NaN,Bloomberg,1,2002-03-01 09:00:00+01:00,NaT,2002-02-28 21:45:00+01:00,2002-03-01 16:00:00+01:00
3,01.03.2002 04:30:00,NaT,UK Monetary Aggregates,Macro Release,UK,Monetary Aggregates,NaN,Bloomberg,1,2002-03-01 10:30:00+01:00,NaT,2002-02-28 21:45:00+01:00,2002-03-01 16:00:00+01:00
4,01.03.2002 06:00:00,NaT,EA Retail Sales & EA Retail Trade,Macro Release,EA,Retail Sales & EA Retail Trade,NaN,Bloomberg,1,2002-03-01 12:00:00+01:00,NaT,2002-02-28 21:45:00+01:00,2002-03-01 16:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51566,31.08.2020 11:35:00,NaT,US Auction Result Bill,Auction,US Result,Bill,CUSIP 9127964F3,https://www.treasurydirect.gov/instit/annceres...,1,2020-08-31 17:30:00+01:00,NaT,2020-08-31 17:15:00+01:00,2020-08-31 18:00:00+01:00
51567,31.08.2020 11:35:00,NaT,US Auction Result Bill,Auction,US Result,Bill,CUSIP 912796TU3,https://www.treasurydirect.gov/instit/annceres...,1,2020-08-31 17:30:00+01:00,NaT,2020-08-31 17:15:00+01:00,2020-08-31 18:00:00+01:00
51568,31.08.2020 20:00:00,NaT,IE Investec Manufacturing PMI,Macro Release,IE,Investec Manufacturing PMI,NaN,Bloomberg,1,2020-09-01 02:00:00+01:00,NaT,2020-08-31 20:45:00+01:00,2020-09-01 15:00:00+01:00
51569,31.08.2020 20:30:00,NaT,JP Markit/JMMA Manufacturing PMI,Macro Release,JP,Markit/JMMA Manufacturing PMI,NaN,Bloomberg,1,2020-09-01 02:30:00+01:00,NaT,2020-08-31 20:45:00+01:00,2020-09-01 15:00:00+01:00


In [3]:
#todo make this faster
print(f"creating {len(df_news["subsubtype"].unique())} event dummie variables")
dummies = {}
for subsubtype, df_type in df_news.groupby("subsubtype"):
    mask = pd.Series(False, index=df.index)
    for _, row in df_type.iterrows():
        mask |= (df.index >= row["window_start"]) & (df.index < row["window_end"])
        dummies[f"D_{subsubtype}"] = mask

df = pd.concat([df, pd.DataFrame(dummies)], axis=1)
df

creating 128 event dummie variables
7047


,open,high,low,close,volume,y^2,day_of_week,hour_of_day,day_1,day_2,...,D_Trade & Current Account Balance,D_Trade Balance,D_Trade Balance Non-Eu (Euros),D_Trade Data (CNY),D_Ulster Bank Construction PMI,D_Unemployment Net,D_Unemployment Rate,D_University of Michigan Surveys,D_Weekly Financial Statement,D_Wholesale Price Index
datetime,,,,,,,,,,,,,,,,,,,,,
2005-03-01 15:30:00+01:00,83.1154,83.4113,83.1154,83.3769,3869200.0,1.779731e-05,1,15,True,False,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 15:45:00+01:00,83.3769,83.4044,83.2805,83.3494,2046100.0,1.088221e-07,1,15,True,False,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 16:00:00+01:00,83.3494,83.5971,83.2874,83.4870,5279800.0,2.720918e-06,1,16,True,False,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 16:15:00+01:00,83.4870,83.5145,83.4044,83.4113,2816800.0,8.229017e-07,1,16,True,False,...,False,False,False,False,False,False,False,False,False,False
2005-03-01 16:30:00+01:00,83.4113,83.5489,83.3906,83.4870,1562300.0,8.229017e-07,1,16,True,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006-02-28 21:00:00+01:00,89.8593,89.9154,89.7263,89.7543,3240400.0,1.190879e-06,1,21,True,False,...,False,False,False,False,False,False,False,False,False,False
2006-02-28 21:15:00+01:00,89.7543,89.9154,89.7473,89.9014,1836600.0,2.681659e-06,1,21,True,False,...,False,False,False,False,False,False,False,False,False,False
2006-02-28 21:30:00+01:00,89.9014,89.9854,89.8453,89.9644,1822900.0,4.907315e-07,1,21,True,False,...,False,False,False,False,False,False,False,False,False,False


In [16]:
# Regression
k = 4

fixed_effects = pd.concat([day_dummies, hour_dummies], axis=1).astype(float)
event_dummies = pd.DataFrame(dummies).astype(float)
lagged_variance = df['y^2'].rolling(window=k).sum()  # assuming k 15-min intervals according to paper




x = pd.concat([event_dummies, fixed_effects, lagged_variance], axis=1)
x.dropna(inplace=True) #drops first 3 rows due to rolling window
x = sm.add_constant(x)
y = df["y^2"].iloc[3:].astype(float)
#model = sm.OLS(y, x).fit(cov_type='cluster', cov_kwds={'groups': df.index.date})
model = sm.OLS(y, x).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    y^2   R-squared:                       0.320
Model:                            OLS   Adj. R-squared:                  0.316
Method:                 Least Squares   F-statistic:                     82.24
Date:                Thu, 10 Jul 2025   Prob (F-statistic):               0.00
Time:                        10:51:12   Log-Likelihood:                 80485.
No. Observations:                7044   AIC:                        -1.609e+05
Df Residuals:                    7003   BIC:                        -1.606e+05
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                                            coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------